# [3.9a] Pipeline Logging — Batch Worker Demo

**Sprint demo:** Execute the actual BrandLens batch worker with three synthetic images, capture job/image-level logs, and verify successful processing, filtering, failure handling, and safe diagnostic logging.

This notebook is intended for **`examples/3_9a_logging_example.ipynb`**. It uses the real `brandlens.api.worker.run_job`, database persistence, filtering/standardization, and `brandlens.logger` from your current branch. 

> **Scope:** A small deterministic analyzer returns a demo score for the valid image and deliberately raises an exception for the failure image. This tests **worker orchestration and logging**, not the quality of CV/VLM scoring. Your `pipeline.py` handles criterion-specific `cv:<criterion>` logs when the real analysis adapter is active.

## 1 · Imports and temporary workspace

All generated images, processed files, and SQLite data live in a temporary directory and are cleaned up after the notebook kernel stops (or you explicitly close the workspace). No project files are modified.

In [1]:
from __future__ import annotations

import io
import logging
import tempfile
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import display
from PIL import Image

# Works when launched from either the repository root or examples/.
_root_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in _root_candidates if (p / "src" / "brandlens").is_dir()), None)

from brandlens.api import db as database
from brandlens.api.worker import ImageAnalysis, run_job

workspace = tempfile.TemporaryDirectory(prefix="brandlens-3-9a-demo-")
BASE = Path(workspace.name)
DB_PATH = BASE / "demo.sqlite3"
RAW_DIR = BASE / "raw"
PROCESSED_DIR = BASE / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)

database.init_db(DB_PATH)
print("Notebook repository:", ROOT)
print("Temporary demo workspace:", BASE)

Notebook repository: f:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\visual-brand-audit
Temporary demo workspace: C:\Users\sonic\AppData\Local\Temp\brandlens-3-9a-demo-9uxzyfep


## 2 · Prepare three synthetic images and a real queued job

- **Valid:** 120 × 100 solid-blue image → demo score of 82.
- **Filtered:** 10 × 10 image → rejected by the project's minimum-image-size filter.
- **Failure:** 120 × 100 solid-red image → analyzer intentionally raises `RuntimeError` with a simulated sensitive path.

Images contain no user photographs. The sensitive path is only a dummy string used to verify that application logs don't reproduce exception messages.

In [2]:
JOB_ID = str(uuid4())
RUBRIC_ID = "default-v1-draft"
CRITERION_IDS = ["exposure_brightness", "contrast"]

samples = [
    ("valid", (120, 100), (50, 100, 160)),
    ("filtered", (10, 10), (110, 110, 110)),
    ("failure", (120, 100), (230, 55, 45)),
]

records = []
labels = {}
with database.get_conn(DB_PATH) as conn:
    database.create_job(conn, job_id=JOB_ID, rubric_id=RUBRIC_ID)
    database.insert_job_criteria(conn, JOB_ID, CRITERION_IDS)
    for i, (label, size, rgb) in enumerate(samples):
        path = RAW_DIR / f"sample_{i}.png"
        Image.new("RGB", size, color=rgb).save(path, "PNG")
        image_id = database.create_image(
            conn,
            job_id=JOB_ID,
            ordinal=i,
            source=f"synthetic:{label}",
            raw_path=str(path),
        )
        records.append((image_id, path))
        labels[image_id] = label

print("Job ID:", JOB_ID)
print("Number of images:", len(records))
display(pd.DataFrame([{"sample": label, "dimensions": f"{size[0]} × {size[1]}", "RGB": str(rgb)} for label, size, rgb in samples]))

Job ID: 0fbb30c5-6428-43a6-aa5e-f196f774151e
Number of images: 3


,sample,dimensions,RGB
0,valid,120 × 100,"(50, 100, 160)"
1,filtered,10 × 10,"(110, 110, 110)"
2,failure,120 × 100,"(230, 55, 45)"


## 3 · Define a controlled analyzer

The worker uses its real `analyze` hook. The analyzer returns a valid `ImageAnalysis` and two criterion-result records on the successful image. On the red image it raises a deliberate exception. This lets us test privacy-safe failure logging without any external model.

In [ ]:
SENSITIVE_EXAMPLE = "/private/customer-images/confidential_photo.png"

def demo_analyze(image: Image.Image, criterion_ids: list[str]) -> ImageAnalysis:
    r, _, b = image.getpixel((0, 0))
    if r > 180 and r > b:
        raise RuntimeError(f"Simulated analyzer error at {SENSITIVE_EXAMPLE}")
    score = 82.0
    criteria = [
        {
            "criterion_id": cid,
            "score": score,
            "weight": 1.0 / len(criterion_ids),
            "source": "cv",
            "availability": "ok",
            "explanation": "Synthetic demo score, not a real CV assessment",
        }
        for cid in criterion_ids
    ]
    return ImageAnalysis(status="scored", overall_score=score, flagged=False, criteria=criteria)

print("Controlled analyzer configured (no VLM or API key required).")

Controlled analyzer configured (no VLM or API key required).


## 4 · Execute the real batch and capture logs

Both the `brandlens.pipeline` logger and `brandlens.api.worker` logger are collected. The handler is detached in `finally` to avoid contaminating later notebook cells.

In [4]:
log_stream = io.StringIO()
handler = logging.StreamHandler(log_stream)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter("%(levelname)s | %(name)s | %(message)s"))
tracked_loggers = [logging.getLogger("brandlens.pipeline"), logging.getLogger("brandlens.api.worker")]
previous_levels = {logger: logger.level for logger in tracked_loggers}

try:
    for logger in tracked_loggers:
        logger.setLevel(logging.INFO)
        logger.addHandler(handler)
    run_job(JOB_ID, records, DB_PATH, PROCESSED_DIR, analyze=demo_analyze)
finally:
    handler.flush()
    for logger in tracked_loggers:
        logger.removeHandler(handler)
        logger.setLevel(previous_levels[logger])

LOG_TEXT = log_stream.getvalue()
print(LOG_TEXT)

INFO | brandlens.pipeline | job=0fbb30c5-6428-43a6-aa5e-f196f774151e image=59fe7112-6264-457c-aa17-466d7a8e91a3 stage=image_loading status=started
INFO | brandlens.pipeline | job=0fbb30c5-6428-43a6-aa5e-f196f774151e image=59fe7112-6264-457c-aa17-466d7a8e91a3 stage=image_loading status=completed duration_ms=10.1
INFO | brandlens.pipeline | job=0fbb30c5-6428-43a6-aa5e-f196f774151e image=59fe7112-6264-457c-aa17-466d7a8e91a3 stage=photo_filter status=started
INFO | brandlens.pipeline | job=0fbb30c5-6428-43a6-aa5e-f196f774151e image=59fe7112-6264-457c-aa17-466d7a8e91a3 stage=photo_filter status=completed duration_ms=0.0
INFO | brandlens.pipeline | job=0fbb30c5-6428-43a6-aa5e-f196f774151e image=59fe7112-6264-457c-aa17-466d7a8e91a3 stage=standardization status=started
INFO | brandlens.pipeline | job=0fbb30c5-6428-43a6-aa5e-f196f774151e image=59fe7112-6264-457c-aa17-466d7a8e91a3 stage=standardization status=completed duration_ms=1.4
INFO | brandlens.pipeline | job=0fbb30c5-6428-43a6-aa5e-f196f

## 5 · Inspect persisted results

This queries the **real** SQLite tables after the worker finishes, so the outcomes are not inferred from log strings.

In [5]:
with database.get_conn(DB_PATH) as conn:
    job = database.get_job(conn, JOB_ID)
    image_rows = database.get_job_images(conn, JOB_ID)
    scores = database.get_job_criterion_scores(conn, JOB_ID)

results = pd.DataFrame([
    {
        "sample": labels[row["image_id"]],
        "image_id": row["image_id"],
        "status": row["status"],
        "reject_reason": row["reject_reason"] or "—",
        "error_code": row["error_code"] or "—",
        "overall_score": row["overall_score"],
        "saved_criteria": len(scores.get(row["image_id"], [])),
    }
    for row in image_rows
])
print("Final job status:", job["status"])
display(results)

Final job status: completed_with_errors


,sample,image_id,status,reject_reason,error_code,overall_score,saved_criteria
0,valid,59fe7112-6264-457c-aa17-466d7a8e91a3,scored,—,—,82.0,2
1,filtered,c749d1b0-a1ab-4c81-8a63-e336da9a2877,filtered_out,too_small,—,NaN,0
2,failure,5f0c7dfd-3448-4968-8263-970526c4c562,failed,—,ANALYSIS_ERROR,NaN,0


## 6 · Verify acceptance criteria

The checks below confirm that image-level log records exist for all three outcomes, successful stages include durations, analysis failure records include an exception **type** but not the exception **message**, and results were saved independently.

In [6]:
statuses = {row["source"].split(":")[-1]: row["status"] for row in image_rows}
assert statuses == {"valid": "scored", "filtered": "filtered_out", "failure": "failed"}, statuses
assert job["status"] == "completed_with_errors"

for image_id, _ in records:
    assert f"job={JOB_ID} image={image_id}" in LOG_TEXT
for stage in ("image_loading", "photo_filter", "standardization", "analysis"):
    assert f"stage={stage} status=started" in LOG_TEXT
    assert f"stage={stage} status=completed" in LOG_TEXT
assert "stage=analysis status=failed" in LOG_TEXT
assert "error_type=RuntimeError" in LOG_TEXT
assert "stage=image status=scored" in LOG_TEXT
assert "stage=image status=filtered_out" in LOG_TEXT
assert "stage=image status=failed" in LOG_TEXT
assert "reason=too_small" in LOG_TEXT
assert "duration_ms=" in LOG_TEXT
assert SENSITIVE_EXAMPLE not in LOG_TEXT, "Sensitive exception detail appeared in application logs!"

checks = pd.DataFrame([
    ("Job/image correlation", "PASS"),
    ("Stage starts and completions", "PASS"),
    ("Stage durations", "PASS"),
    ("Filtered image reason", "PASS"),
    ("Failure event and exception type", "PASS"),
    ("Sensitive data absent from logs", "PASS"),
    ("Independent persisted outcomes", "PASS"),
], columns=["acceptance check", "result"])
display(checks)
print("All [3.9a] demo assertions passed.")

,acceptance check,result
0,Job/image correlation,PASS
1,Stage starts and completions,PASS
2,Stage durations,PASS
3,Filtered image reason,PASS
4,Failure event and exception type,PASS
5,Sensitive data absent from logs,PASS
6,Independent persisted outcomes,PASS


All [3.9a] demo assertions passed.


## 7 · Summarize logs by stage (optional visual for the Sprint video)

This summary parses only the structured `brandlens.pipeline` messages and reports the number of recorded start, completion, failure, and final-outcome events.

In [7]:
import re

parsed = []
for line in LOG_TEXT.splitlines():
    if "| brandlens.pipeline | " not in line:
        continue
    stage_match = re.search(r"\bstage=([^ ]+)", line)
    status_match = re.search(r"\bstatus=([^ ]+)", line)
    if stage_match and status_match:
        parsed.append({"stage": stage_match.group(1), "status": status_match.group(1)})

if parsed:
    overview = (
        pd.DataFrame(parsed)
        .groupby(["stage", "status"])
        .size()
        .reset_index(name="count")
        .sort_values(["stage", "status"])
    )
    display(overview)
else:
    print("No structured pipeline events were captured.")

,stage,status,count
0,analysis,completed,1
1,analysis,failed,1
2,analysis,started,2
3,image,failed,1
4,image,filtered_out,2
5,image,scored,1
6,image_loading,completed,3
7,image_loading,started,3
8,photo_filter,completed,3
9,photo_filter,filtered_out,1


## 8 · Sprint demonstration notes

**Suggested narration (roughly 45 seconds):**

> My task [3.9a] adds privacy-conscious logging to image preprocessing and CV analysis. This notebook invokes the actual batch worker using three synthetic images. The logs identify each job and image, record when loading, filtering, standardization, and analysis start and finish, and report execution time. A small image is filtered out, while a deliberately failing analyzer demonstrates error handling. The final outcomes are stored independently in SQLite, and our assertions verify that sensitive exception details do not appear in the logs. Individual CV criterion logs are also instrumented in `pipeline.py` when the full analyzer is connected.

**Limitations:** The success score here is an intentional demonstration value; it is **not** produced by real CV analysis. The dummy secret may still be stored in database error details by the current worker design; the privacy assertion here specifically covers **application logs**. Consider a separate review of database/API error sanitization.

**Cleanup (optional):** When finished, run `workspace.cleanup()` to remove all temporary files. If you rerun the notebook after cleanup, restart from cell 1.